1: Import Libraries & Generate Dataset

In [9]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv('health_data.csv')

print("Dataset Loaded Successfully!")
display(df.head())


Dataset Loaded Successfully!


,record_id,age_group,age,weight,gender,region,smoking_status,exercise_frequency,bmi,blood_pressure,diabetes,hypertension,cholesterol_level,glucose_level,visit_date
0,1001,18-25,22,65,Female,North,Non-Smoker,Daily,22.1,118.5,False,False,165.2,85.4,2024-01-12
1,1002,36-45,40,85,Male,South,Smoker,Rarely,28.9,135.2,True,True,215.8,126.5,2024-02-15
2,1003,60+,68,92,Male,East,Former Smoker,Never,31.5,145.0,True,True,245.3,158.0,2024-03-08
3,1004,26-35,30,70,Female,West,Non-Smoker,Weekly,24.2,122.1,False,False,180.5,92.3,2024-01-22
4,1005,46-60,55,88,Male,North,Smoker,Never,30.1,142.5,True,True,230.1,148.9,2024-02-27


1 - Formulate Hypotheses

In [10]:
print("--- TASK 1: HYPOTHESES FORMULATION ---")
print("\n1. Hypothesis for T-Test (Group Comparison):")
print("   H0: Smoking status has NO effect on BMI. (Mean BMI of Smokers = Mean BMI of Non-Smokers)")
print("   H1: Smoking status AFFECTS BMI. (Mean BMI of Smokers != Mean BMI of Non-Smokers)")

print("\n2. Hypothesis for Chi-Square Test (Categorical Association):")
print("   H0: Region and Diabetes prevalence are independent.")
print("   H1: Region and Diabetes prevalence are dependent (related).")

print("\n3. Hypothesis for ANOVA (Multiple Group Comparison):")
print("   H0: The mean Glucose Level is the same across all Age Groups.")
print("   H1: At least one Age Group has a significantly different mean Glucose Level.")

--- TASK 1: HYPOTHESES FORMULATION ---

1. Hypothesis for T-Test (Group Comparison):
   H0: Smoking status has NO effect on BMI. (Mean BMI of Smokers = Mean BMI of Non-Smokers)
   H1: Smoking status AFFECTS BMI. (Mean BMI of Smokers != Mean BMI of Non-Smokers)

2. Hypothesis for Chi-Square Test (Categorical Association):
   H0: Region and Diabetes prevalence are independent.
   H1: Region and Diabetes prevalence are dependent (related).

3. Hypothesis for ANOVA (Multiple Group Comparison):
   H0: The mean Glucose Level is the same across all Age Groups.
   H1: At least one Age Group has a significantly different mean Glucose Level.


2 - Calculate Confidence Intervals

In [11]:
print("--- TASK 2: CONFIDENCE INTERVALS (95%) ---")

def get_ci(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data) 
    margin = sem * stats.t.ppf((1 + confidence) / 2., n-1)
    return mean, mean - margin, mean + margin

age_m, age_l, age_h = get_ci(df['age'])
print(f"Age: Mean = {age_m:.2f} | 95% CI = [{age_l:.2f}, {age_h:.2f}]")

wt_m, wt_l, wt_h = get_ci(df['weight'])
print(f"Weight: Mean = {wt_m:.2f} | 95% CI = [{wt_l:.2f}, {wt_h:.2f}]")

--- TASK 2: CONFIDENCE INTERVALS (95%) ---
Age: Mean = 43.22 | 95% CI = [38.24, 48.20]
Weight: Mean = 76.86 | 95% CI = [73.34, 80.38]


3 & 4 - T-Test, Critical Value, & P-Value

In [14]:
print("--- TASK 3 & 4: T-TEST, CRITICAL VALUE & P-VALUE ---")

group_smoker = df[df['smoking_status'] == 'Smoker']['bmi']
group_nonsmoker = df[df['smoking_status'] == 'Non-Smoker']['bmi']


t_stat, p_val = stats.ttest_ind(group_smoker, group_nonsmoker)


alpha = 0.05
df_freedom = len(group_smoker) + len(group_nonsmoker) - 2
critical_value = stats.t.ppf(1 - alpha/2, df_freedom)

print(f"Comparison: BMI of Smokers vs Non-Smokers")
print(f"T-Statistic: {t_stat:.4f}")
print(f"Critical Value: +/- {critical_value:.4f}")
print(f"P-Value: {p_val:.4e}") 

print("\n--- Interpretation ---")
if abs(t_stat) > critical_value:
    print("Decision: Reject Null Hypothesis (H0).")
    print("Conclusion: There is a significant difference in BMI between Smokers and Non-Smokers.")
else:
    print("Decision: Accept Null Hypothesis (H0).")
    print("Conclusion: No significant difference found.")


--- TASK 3 & 4: T-TEST, CRITICAL VALUE & P-VALUE ---
Comparison: BMI of Smokers vs Non-Smokers
T-Statistic: 1.5337
Critical Value: +/- 2.0244
P-Value: 1.3339e-01

--- Interpretation ---
Decision: Accept Null Hypothesis (H0).
Conclusion: No significant difference found.


5 - Chi-Square Test

In [15]:
print("--- TASK 5: CHI-SQUARE TEST ---")

contingency_table = pd.crosstab(df['region'], df['diabetes'])
print("Contingency Table:\n", contingency_table)

chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p:.4f}")

print("\n--- Interpretation ---")
if p < 0.05:
    print("Decision: Reject H0.")
    print("Conclusion: There is a significant association between Region and Diabetes.")
else:
    print("Decision: Accept H0.")
    print("Conclusion: Region and Diabetes appear independent (no relationship).")

--- TASK 5: CHI-SQUARE TEST ---
Contingency Table:
 diabetes  False  True 
region                
East          4      8
North         6      7
South         9      4
West         10      2

Chi-Square Statistic: 7.5807
P-Value: 0.0555

--- Interpretation ---
Decision: Accept H0.
Conclusion: Region and Diabetes appear independent (no relationship).


6 - ANOVA Test

In [16]:
print("--- TASK 6: ONE-WAY ANOVA ---")

groups = [df[df['age_group'] == g]['glucose_level'] for g in df['age_group'].unique()]


f_stat, p_val = stats.f_oneway(*groups)

print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_val:.4e}")

print("\n--- Interpretation ---")
if p_val < 0.05:
    print("Decision: Reject H0.")
    print("Conclusion: Age Groups significantly differ in Glucose Levels.")
else:
    print("Decision: Accept H0.")
    print("Conclusion: No significant difference in Glucose Levels across Age Groups.")



--- TASK 6: ONE-WAY ANOVA ---
F-Statistic: 21.2667
P-Value: 6.6816e-10

--- Interpretation ---
Decision: Reject H0.
Conclusion: Age Groups significantly differ in Glucose Levels.


7 & 8 - Covariance, Correlation & Final Results

In [17]:
print("--- TASK 7: COVARIANCE & CORRELATION ---")

continuous_vars = ['age', 'bmi', 'glucose_level', 'weight']

cov_matrix = df[continuous_vars].cov()
print("Covariance Matrix:\n", cov_matrix)

corr_matrix = df[continuous_vars].corr()
print("\nCorrelation Matrix:\n", corr_matrix)

r_val = corr_matrix.loc['age', 'glucose_level']
print(f"\nCorrelation (Age vs Glucose): {r_val:.4f}")

if r_val > 0.5:
    print("Interpretation: Strong Positive Correlation.")
elif r_val > 0:
    print("Interpretation: Weak/Moderate Positive Correlation.")
elif r_val < 0:
    print("Interpretation: Negative Correlation.")


print("\n--- TASK 8: FINAL SUMMARY OF RESULTS ---")
print("1. T-Test: Tested if Smoking impacts BMI. Result printed in Task 4.")
print("2. Chi-Square: Tested if Region impacts Diabetes. Result printed in Task 5.")
print("3. ANOVA: Tested if Age impacts Glucose. Result printed in Task 6.")
print("4. Correlation: Analyzed relationships between Age, BMI, and Glucose.")

--- TASK 7: COVARIANCE & CORRELATION ---
Covariance Matrix:
                       age        bmi  glucose_level      weight
age            307.399592  24.056980     322.079673   85.153878
bmi             24.056980  18.650220      87.549380   53.132571
glucose_level  322.079673  87.549380     617.546433  262.441878
weight          85.153878  53.132571     262.441878  153.510612

Correlation Matrix:
                     age       bmi  glucose_level    weight
age            1.000000  0.317722       0.739225  0.391998
bmi            0.317722  1.000000       0.815786  0.993001
glucose_level  0.739225  0.815786       1.000000  0.852372
weight         0.391998  0.993001       0.852372  1.000000

Correlation (Age vs Glucose): 0.7392
Interpretation: Strong Positive Correlation.

--- TASK 8: FINAL SUMMARY OF RESULTS ---
1. T-Test: Tested if Smoking impacts BMI. Result printed in Task 4.
2. Chi-Square: Tested if Region impacts Diabetes. Result printed in Task 5.
3. ANOVA: Tested if Age impacts G